# Testing

In [1]:
import duckdb
import plotly.io as pio
import os

import fastf1 as ff1
import pandas as pd
from scipy.spatial import cKDTree

pio.renderers.default = "notebook" 

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)


# Driver minisectors/sectors

In [2]:
file_path = os.path.abspath("data/driver_timing/2025_1_5_Australian_Grand_Prix_R.parquet")

In [3]:
drivers_telemetry = duckdb.sql(f"SELECT * FROM '{file_path}'").df()

In [4]:
#print(drivers_telemetry.Sector.unique())
#print(drivers_telemetry.CornerArea.unique())
#print(drivers_telemetry.DistanceFromPreviousCorner.unique())
#print(drivers_telemetry.DistanceFromPreviousCorner.unique())
print(drivers_telemetry.NextCorner.unique())
#print(drivers_telemetry.PreviousCorner.unique())

[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. nan]


In [5]:
drivers_telemetry.loc[(drivers_telemetry.DriverNumber=='1')].head(50)

,DriverNumber,Driver,Team,Stint,LapNumber,Time,Speed,RPM,nGear,Throttle,...,SpeedFL,SpeedST,FreshTyre,Year,EventName,SessionName,Location,year_1,event,session
0,1,VER,Red Bull Racing,4.0,6.0,214000000,243.0,10828.0,6,100.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
1,1,VER,Red Bull Racing,4.0,6.0,394000000,245.0,10931.0,6,100.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
2,1,VER,Red Bull Racing,4.0,6.0,553000000,245.0,10931.0,6,100.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
3,1,VER,Red Bull Racing,4.0,6.0,893000000,251.0,11059.0,6,100.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
4,1,VER,Red Bull Racing,4.0,6.0,1073000000,251.0,11059.0,6,100.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
5,1,VER,Red Bull Racing,4.0,6.0,1393000000,232.0,10198.0,6,0.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
6,1,VER,Red Bull Racing,4.0,6.0,1714000000,223.0,10425.0,6,0.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
7,1,VER,Red Bull Racing,4.0,6.0,2114000000,183.0,10264.0,5,0.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
8,1,VER,Red Bull Racing,4.0,6.0,2274000000,183.0,10264.0,5,0.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R
9,1,VER,Red Bull Racing,4.0,6.0,2494000000,161.0,11255.0,4,0.0,...,208.0,127.0,False,2025,Australian Grand Prix,Race,Melbourne,2025,Australian Grand Prix,R


In [8]:
df=pd.read_csv('data/predictions/ssot/2025_qualifying.csv')

In [12]:
df.loc[df.WeekendId=='2025_01'].sort_values(by='Position')

,WeekendId,Season,RoundNumber,EventName,SessionName,SessionStart,DriverNumber,Abbreviation,DriverId,BroadcastName,TeamName,GridPosition,ClassifiedPosition,Status,Q1,Q2,Q3,BestLapTime,BestLapSpeed,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,Time,Points
11,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,4,NOR,norris,L NORRIS,McLaren,NaN,NaN,NaN,0 days 00:01:15.912000,0 days 00:01:15.415000,0 days 00:01:15.096000,NaN,NaN,FF8000,mclaren,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,NaN,1.0,NaN,NaN
18,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,81,PIA,piastri,O PIASTRI,McLaren,NaN,NaN,NaN,0 days 00:01:16.062000,0 days 00:01:15.468000,0 days 00:01:15.180000,NaN,NaN,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,https://media.formula1.com/d_driver_fallback_i...,NaN,2.0,NaN,NaN
0,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,1,VER,max_verstappen,M VERSTAPPEN,Red Bull Racing,NaN,NaN,NaN,0 days 00:01:16.018000,0 days 00:01:15.565000,0 days 00:01:15.481000,NaN,NaN,3671C6,red_bull,Max,Verstappen,Max Verstappen,https://media.formula1.com/d_driver_fallback_i...,NaN,3.0,NaN,NaN
16,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,63,RUS,russell,G RUSSELL,Mercedes,NaN,NaN,NaN,0 days 00:01:15.971000,0 days 00:01:15.798000,0 days 00:01:15.546000,NaN,NaN,27F4D2,mercedes,George,Russell,George Russell,https://media.formula1.com/d_driver_fallback_i...,NaN,4.0,NaN,NaN
6,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,22,TSU,tsunoda,Y TSUNODA,Racing Bulls,NaN,NaN,NaN,0 days 00:01:16.225000,0 days 00:01:16.009000,0 days 00:01:15.670000,NaN,NaN,6692FF,rb,Yuki,Tsunoda,Yuki Tsunoda,https://media.formula1.com/d_driver_fallback_i...,NaN,5.0,NaN,NaN
7,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,23,ALB,albon,A ALBON,Williams,NaN,NaN,NaN,0 days 00:01:16.245000,0 days 00:01:16.017000,0 days 00:01:15.737000,NaN,NaN,64C4FF,williams,Alexander,Albon,Alexander Albon,https://media.formula1.com/d_driver_fallback_i...,NaN,6.0,NaN,NaN
4,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,16,LEC,leclerc,C LECLERC,Ferrari,NaN,NaN,NaN,0 days 00:01:16.029000,0 days 00:01:15.827000,0 days 00:01:15.755000,NaN,NaN,E80020,ferrari,Charles,Leclerc,Charles Leclerc,https://media.formula1.com/d_driver_fallback_i...,NaN,7.0,NaN,NaN
12,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,44,HAM,hamilton,L HAMILTON,Ferrari,NaN,NaN,NaN,0 days 00:01:16.213000,0 days 00:01:15.919000,0 days 00:01:15.973000,NaN,NaN,E80020,ferrari,Lewis,Hamilton,Lewis Hamilton,https://media.formula1.com/d_driver_fallback_i...,NaN,8.0,NaN,NaN
1,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,10,GAS,gasly,P GASLY,Alpine,NaN,NaN,NaN,0 days 00:01:16.328000,0 days 00:01:16.112000,0 days 00:01:15.980000,NaN,NaN,0093CC,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,NaN,9.0,NaN,NaN
14,2025_01,2025,1,Australian Grand Prix,Qualifying,0:12:28.580000,55,SAI,sainz,C SAINZ,Williams,NaN,NaN,NaN,0 days 00:01:16.360000,0 days 00:01:15.931000,0 days 00:01:16.062000,NaN,NaN,64C4FF,williams,Carlos,Sainz,Carlos Sainz,https://media.formula1.com/d_driver_fallback_i...,NaN,10.0,NaN,NaN


In [6]:
sched = ff1.get_event_schedule(2025)
len(sched.loc[sched.EventFormat=='testing'])

1

In [ ]:
sched = ff1.get_event_schedule(2025)
len(sched.loc[sched.EventFormat=='testing'])

for i in range(1,len(sched.loc[sched.EventFormat=='testing'])+1):
    ff1.get_testing_event(2025, i)
    print(i)

1


In [16]:
for i in range(1,4):
    ff1.get_testing_session(2025, 1,i)
    print(i)


1
2
3


In [15]:
sched.loc[sched.EventFormat=='testing']

,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,Session2,Session2Date,Session2DateUtc,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
0,0,Bahrain,Sakhir,FORMULA 1 ARAMCO PRE-SEASON TESTING 2025,2025-02-28,Pre-Season Testing,testing,Practice 1,2025-02-26 10:00:00+03:00,2025-02-26 07:00:00,Practice 2,2025-02-27 10:00:00+03:00,2025-02-27 07:00:00,Practice 3,2025-02-28 10:00:00+03:00,2025-02-28 07:00:00,None,NaT,NaT,None,NaT,NaT,True
